# Routing MultiAgent

In [1]:
import os
import json
from dotenv import load_dotenv
from groq import Groq

The application will automatically select the best Agent to answer based on the complexity of the question.

Question Types:

- **Factual**: Short and direct questions.
- **Research**: Questions that require detailed answers.
- **General**: Other queries.

Generative AI makes mistakes. **ALWAYS** use your knowledge to verify the answers.

In [5]:
load_dotenv()

with open('groq_models.json', 'r') as file:
    models_config = json.load(file)

In [6]:
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

In [9]:
class IntentionAgent:
    @staticmethod
    def evaluate_intention(query):
        if len(query.split()) <= 10 and 'Provide' in query:
            return 'factual'

        elif '?' in query:
            return 'research'

        else:
            return 'general'

In [12]:
class InformationAgent:
    @staticmethod
    def search_information(query, intention):
        config = models_config[intention]
        model = config["model"]

        custom_prompt = config["prompt"].format(query=query)

        response = client.chat.completions.create(
            messages = [{"role": "user", "content": custom_prompt}],
            model = model,
        )

        return response.choices[0].message.content, model

In [13]:
class ResponseAgent:
    @staticmethod
    def formulate_response(information, intention):

        if intention == 'factual':
            return f"Fast response: {information}"

        elif intention == 'research':
            return f"Detailed response: {information}"

        else:
            return f"General response: {information}"

In [14]:
def get_response(query):
    intention = IntentionAgent.evaluate_intention(query)

    information, used_model = InformationAgent.search_information(query, intention)

    final_response = ResponseAgent.formulate_response(information, intention)

    return final_response, used_model

### Factual:
* Provide what is the Spain capital
* Provide who invented the phone
* Provide when Microsoft was established

### Research:
* How does Artificial Intelligence work in autonomous vehicles?
* What are the economical impacts of the climate changes?
* What are the most recent advances in Alzheimer treatment?

### General:
* Give me tips and suggestions to learn a new language
* Briefly explain the concept of circular economy
* Tell me something interesting about astronomy

In [20]:
query = "Give me tips and suggestions to learn a new language"

response, used_model = get_response(query)

In [21]:
print(response)

print(used_model)

General response: Here are some tips and suggestions to help you learn a new language:

1. **Set achievable goals**: Define your motivation and set realistic goals, such as passing a language test or conversing with native speakers.
2. **Immerse yourself in the language**: Listen to music, watch TV shows and movies, read books and newspapers, and speak with native speakers to get used to the sounds, rhythms, and grammar.
3. **Focus on grammar and vocabulary**: Learn the basics of grammar and build your vocabulary gradually. Start with common phrases and expressions, and then move on to more complex grammar rules.
4. **Practice consistently**: Set aside time each day to practice speaking, writing, and listening to the language. Use language learning apps, flashcards, or language exchange websites to stay consistent.
5. **Learn colloquial expressions and idioms**: Idioms and colloquial expressions can make your language sound more natural and help you communicate effectively.
6. **Use la